In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import os

print("CPU cores (os.cpu_count):", os.cpu_count())


CPU cores (os.cpu_count): 12


In [4]:
!pip install -q findspark
import findspark
findspark.init()

In [3]:

from pyspark.sql import SparkSession
from pyspark import SparkContext, SparkConf

spark = (
    SparkSession.builder
    .appName("RandomForest_PCA_K25")
    .master("local[*]")
    .config('spark.ui.port', '4050')
    .config("spark.driver.memory", "35g")
    .config("spark.driver.memoryOverhead", "4g")

    .config("spark.sql.shuffle.partitions", "16")
    .config("spark.default.parallelism", "16")

    .config("spark.serializer", "org.apache.spark.serializer.KryoSerializer")

    .getOrCreate()
)

spark


In [ ]:
# spark.stop()

In [6]:
!pip install -q ngrok


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.8/3.8 MB 67.3 MB/s eta 0:00:00


In [7]:
!wget -q https://bin.equinox.io/c/bNyj1mQVY4c/ngrok-v3-stable-linux-amd64.tgz
!tar -xzf ngrok-v3-stable-linux-amd64.tgz
!chmod +x ngrok
get_ipython().system_raw('!./ngrok http 4050 &')


In [8]:
!./ngrok authtoken 372K2OENTZEIyObJBECpiQP0wUR_BgEkQykJerjtRoPQJsoH

Authtoken saved to configuration file: /root/.config/ngrok/ngrok.yml


In [9]:
!nohup ./ngrok http 4050 --log=stdout --log-format=logfmt > ngrok.log 2>&1 &


In [4]:
parquet_path_raw = "/content/drive/MyDrive/bigdata/cic_ddos_raw.parquet"
parquet_path_25 = r"/content/drive/MyDrive/bigdata/cic_ddos_pca_25.parquet"
parquet_path_21 = r"/content/drive/MyDrive/bigdata/cic_ddos_pca_21.parquet"
parquet_path_27 = r"/content/drive/MyDrive/bigdata/cic_ddos_rf_27.parquet"

df_feature_raw = spark.read.parquet(parquet_path_raw).cache()
df_feature_25 = spark.read.parquet(parquet_path_25).cache()
df_feature_21 = spark.read.parquet(parquet_path_21).cache()
df_feature_27 = spark.read.parquet(parquet_path_27).cache()

df_feature_25.printSchema(), df_feature_21.printSchema(), df_feature_27.printSchema(), df_feature_raw.printSchema()

root
 |-- pca_k25: vector (nullable = true)
 |-- label: string (nullable = true)

root
 |-- pca_k21: vector (nullable = true)
 |-- label: string (nullable = true)

root
 |-- label: string (nullable = true)
 |-- rf_27: vector (nullable = true)

root
 |-- features: vector (nullable = true)
 |-- label: string (nullable = true)



(None, None, None, None)

In [5]:
from pyspark.ml.feature import StringIndexer
label_indexer = StringIndexer(
    inputCol="label",
    outputCol="label_index",
    handleInvalid="skip"
)
df_indexed_25 = label_indexer.fit(df_feature_25).transform(df_feature_25)
df_indexed_21 = label_indexer.fit(df_feature_21).transform(df_feature_21)
df_indexed_27 = label_indexer.fit(df_feature_27).transform(df_feature_27)
df_indexed_raw = label_indexer.fit(df_feature_raw).transform(df_feature_raw)


In [6]:
from pyspark.sql.functions import lit

def split_train_test(df_indexed):
  fractions = (
      df_indexed
      .select("label")
      .distinct()
      .withColumn("fraction", lit(0.02))
      .rdd.collectAsMap()
  )

  test_df = df_indexed.sampleBy(
      "label",
      fractions,
      seed=42
  )
  return test_df, df_indexed.subtract(test_df)

test_df_25, train_df_25 = split_train_test(df_indexed_25)
test_df_21, train_df_21 = split_train_test(df_indexed_21)
test_df_27, train_df_27 = split_train_test(df_indexed_27)
test_df_raw, train_df_raw = split_train_test(df_indexed_raw)

In [7]:
from pyspark.sql import functions as F

def weight_df(df):
  label_counts = df.groupBy("label").count()

  stats = label_counts.agg(
      F.sum("count").alias("total_train"),
      F.count("label").alias("num_labels")
  ).first()

  total_train = float(stats["total_train"])
  num_labels = float(stats["num_labels"])

  weights = (
      label_counts
      .withColumn(
          "weight",
          F.lit(float(total_train)) / (F.col("count") * F.lit(float(num_labels)))
      )
      .select("label", "weight")
  )

  df_w  = df.join(weights, on="label", how="left")
  return df_w

# train_w_raw = weight_df(train_df_raw)
test_w_raw = weight_df(test_df_raw)

# train_w_25 = weight_df(train_df_25)
test_w_25 = weight_df(test_df_25)

# train_w_21 = weight_df(train_df_21)
test_w_21 = weight_df(test_df_21)

# train_w_27 = weight_df(train_df_27)
test_w_27 = weight_df(test_df_27)
# train_w.select("label", "weight").distinct().show()



In [ ]:
# print("Train distribution")
# train_df.groupBy("label_index").count().orderBy("label_index").show()
# print("Test distribution")
# test_df.groupBy("label_index").count().orderBy("label_index").show()


In [11]:
from pyspark.ml.classification import RandomForestClassifier
def rf_init(feauture_name):
  rf = RandomForestClassifier(
      labelCol="label_index",
      featuresCol=feauture_name,
      weightCol="weight",
      numTrees=100,
      maxDepth=14,
      maxBins=48,
      featureSubsetStrategy="sqrt",
      subsamplingRate=0.85,
      seed=42
  )
  return rf

rf = rf_init("features")
rf_model_raw = rf.fit(test_w_raw)

rf = rf_init("pca_k25")
rf_model_25 = rf.fit(test_w_25)

rf = rf_init("pca_k21")
rf_model_21 = rf.fit(test_w_21)

rf = rf_init("rf_27")
rf_model_27 = rf.fit(test_w_27)



In [12]:
rf_model_raw_path = r"/content/drive/MyDrive/bigdata/model/rf_cic_ddos_2019_raw"
rf_model_raw.write().overwrite().save(rf_model_raw_path)

rf_model_25_path = r"/content/drive/MyDrive/bigdata/model/rf_cic_ddos_2019_pca_k25"
rf_model_25.write().overwrite().save(rf_model_25_path)

rf_model_21_path = r"/content/drive/MyDrive/bigdata/model/rf_cic_ddos_2019_pca_k21"
rf_model_21.write().overwrite().save(rf_model_21_path)

rf_model_27_path = r"/content/drive/MyDrive/bigdata/model/rf_cic_ddos_2019_rf_k27"
rf_model_27.write().overwrite().save(rf_model_27_path)



In [ ]:
spark.stop()